# W7D5_LAB_Micromanage_your_agents
- Week 7 / Day 5
- Student: Andreas Papachristophorou
- Course: AI Consulting & Integration 2026-07
- Date: 2026-08-23
---

In [1]:
%pip install -q langsmith openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

from dotenv import load_dotenv
from langsmith import Client

load_dotenv()

client = Client()

print("LangSmith endpoint:", os.getenv("LANGSMITH_ENDPOINT"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))
print("LangSmith key found:", bool(os.getenv("LANGSMITH_API_KEY")))
print("OpenAI key found:", bool(os.getenv("OPENAI_API_KEY")))

LangSmith endpoint: https://eu.api.smith.langchain.com
LangSmith project: eu-ets-fueleu-evaluation
LangSmith key found: True
OpenAI key found: True


In [ ]:
# Validate the JSON
import json

with open("eu_ets_fueleu_examples.json", "r", encoding="utf-8") as file:
    examples = json.load(file) # opens the JSON file and converts it into a Python list.

required_keys = {"inputs", "outputs", "metadata"}

print("Number of examples:", len(examples)) # counts dataset rows; it should be 12.
print("All examples have required keys:", all(
    required_keys == set(example.keys()) # checks that every example has exactly inputs, outputs, and metadata.
    for example in examples
))

print("Regulations:", sorted({
    example["metadata"]["regulation"]
    for example in examples
}))

Number of examples: 12
All examples have required keys: True
Regulations: ['EU ETS', 'FuelEU Maritime']


In [3]:
# Import the LangSmith Client class so Python can communicate with LangSmith.
from langsmith import Client

# Create a client using your LangSmith API key from the environment variables.
client = Client()

# Choose a clear, reusable name for your dataset in LangSmith.
DATASET_NAME = "eu-ets-fueleu-maritime-qa"

# Add a short explanation so the dataset is understandable in the LangSmith UI.
DATASET_DESCRIPTION = (
    "Question-answer evaluation dataset covering EU ETS maritime "
    "and FuelEU Maritime requirements."
)

# Get the names of datasets that already exist in your LangSmith workspace.
# This prevents accidentally uploading the same 12 examples twice.
existing_names = {dataset.name for dataset in client.list_datasets()}

# Only create and upload the dataset if that name does not already exist.
if DATASET_NAME in existing_names:
    # Tell us that the dataset already exists instead of creating a duplicate.
    print(f"Dataset already exists: {DATASET_NAME}")

else:
    # Create the empty dataset folder in LangSmith and save its details in 'dataset'.
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description=DATASET_DESCRIPTION
    )

    # Upload all examples at once.
    # The list comprehensions pull the matching inputs, expected answers,
    # and metadata from every example in the 'examples' list.
    client.create_examples(
        inputs=[example["inputs"] for example in examples],
        outputs=[example["outputs"] for example in examples],
        metadata=[example["metadata"] for example in examples],
        dataset_id=dataset.id  # Connect the uploaded examples to the new dataset.
    )

    # Confirm that the upload completed and display the number uploaded.
    print(f"Created dataset and uploaded {len(examples)} examples.")

Created dataset and uploaded 12 examples.


In [ ]:
# Import the OpenAI client and LangSmith tools for tracing.
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

# Wrap the OpenAI client so its model calls are automatically traced in LangSmith.
openai_client = wrap_openai(OpenAI())

# Label this function in LangSmith and record each time it runs.
@traceable(name="maritime-regulation-qa")
def answer_maritime_question(inputs: dict) -> dict:
    # Get the source information and question from one dataset example.
    context = inputs["context"]
    question = inputs["question"]

    # Ask the LLM to answer using only the supplied context.
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer maritime-regulation questions using only the "
                    "provided context. Give a concise, factual answer. "
                    "If the context does not contain the answer, say "
                    "'The provided context does not contain the answer.'"
                )
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {question}"
            }
        ]
    )

    # Return a dictionary whose 'answer' key matches the dataset output format.
    return {"answer": response.choices[0].message.content.strip()}